In [0]:
-- 1. Read, Type, Trim, Deduplicate, and Impute Missing Values
CREATE TEMPORARY LIVE VIEW stg_regions_typed AS
SELECT DISTINCT
  CAST(TRIM(region_id) AS INT) AS region_id,
  TRIM(sales_district) AS sales_district,
  
  -- Impute missing sales_region using Window Function based on sales_district
  COALESCE(
    TRIM(sales_region),
    MAX(TRIM(sales_region)) OVER (PARTITION BY TRIM(sales_district))
  ) AS sales_region,
  
  ingested_timestamp,
  batch_date
FROM mavenmarket.bronze.regions;

-- 2. Run Generic KPI Checks (Completeness, Validity)
CREATE TEMPORARY LIVE VIEW stg_regions_checked AS
SELECT 
  *,
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN region_id IS NULL THEN 'Completeness: region_id is null; ' ELSE '' END AS err_region_id,
  
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  -- Per your rules, we keep it if sales_district is null, but we log a warning for the dashboard
  CASE WHEN sales_district IS NULL THEN 'Completeness: sales_district is null; ' ELSE '' END AS warn_district,
  CASE WHEN sales_region IS NULL THEN 'Completeness: sales_region could not be imputed and is null; ' ELSE '' END AS warn_region
FROM stg_regions_typed;

-- 3. Consolidate Flags
CREATE TEMPORARY LIVE VIEW stg_regions_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_region_id)) AS hard_failures,
  TRIM(CONCAT(warn_district, warn_region)) AS dq_flags
FROM stg_regions_checked;

-- 4. Write Target Tables (Split the rows)
-- CLEAN / WARN TABLE
CREATE LIVE TABLE silver_regions
COMMENT "Cleaned regions dimension with imputed sales regions and warn-level DQ flags."
AS SELECT 
  * EXCEPT(err_region_id, warn_district, warn_region, hard_failures)
FROM stg_regions_flagged
WHERE hard_failures = '';

-- QUARANTINE TABLE
CREATE LIVE TABLE quarantine_regions
COMMENT "Quarantined regions failing hard checks (e.g. missing region_id)"
AS SELECT 
  * EXCEPT(err_region_id, warn_district, warn_region, dq_flags),
  hard_failures AS quarantine_reasons
FROM stg_regions_flagged
WHERE hard_failures != '';